# 07 · Inverse Problems — Discovering Physical Parameters from Data

**Level:** Advanced
**Prerequisites:** `03_heat_equation`

---

## 🧭 Big Picture: Forward vs Inverse Problems

Every physics simulation you have ever run is a **forward problem**: you *know* the equation, you *know* all the parameters (conductivity, viscosity, reaction rate, …), and you compute the solution field. The arrow goes:

> **Known equation + known parameters → compute field**

An **inverse problem** flips this arrow:

> **Observed (noisy, sparse) field data → discover unknown parameters**

### Real-world examples of inverse problems
| Field | What is measured | What is unknown |
|---|---|---|
| Geophysics | Seismic waves at surface sensors | Sub-surface rock stiffness |
| Biomedical | Blood-flow velocity (MRI) | Vessel-wall elasticity, viscosity |
| Chemical engineering | Concentration time-series | Reaction rate constants |
| Structural health | Vibration frequencies | Material damage / cracks |
| Climate science | Surface temperature anomalies | Deep-ocean heat diffusivity |

In all of these, you have *more equations than unknowns* at any one location but the data is *sparse and noisy*. Classical approaches (least-squares regression, Bayesian inference, adjoint methods) work but require either a huge amount of data, expensive PDE solvers in the loop, or carefully derived adjoint equations. **PINNs do it almost for free.**

---

## 🔑 The PINN Inverse Problem Trick

Remember the standard PINN recipe from earlier notebooks:

```
θ* = argmin_{θ} [ L_physics(θ) + L_BC(θ) + L_IC(θ) ]
```

For an inverse problem we simply **add the unknown physical constant as an extra trainable variable** alongside the network weights:

```
θ*, α* = argmin_{θ, α} [ L_physics(θ, α) + L_data(θ) ]
```

- `L_physics` penalises violation of the PDE — it *depends on α* because α appears in the equation.
- `L_data` penalises mismatch between the network prediction and the sparse sensor readings.
- The optimiser (Adam) adjusts **both** the neural-network weights *and* the physical parameter simultaneously.

The physics loss acts as a **powerful regulariser**: the network cannot overfit the noisy data without violating the PDE, and the PDE cannot be satisfied unless α is close to its true value. Data and physics work together to constrain the solution.

---

## Why PINNs excel at this compared to alternatives

| Method | Data needed | Needs adjoint? | Handles noise? | Continuous field? |
|---|---|---|---|---|
| Classical least-squares | ≫ lots | No | Poorly | No |
| Bayesian MCMC | Moderate | No | Yes | No |
| Adjoint PDE inversion | Moderate | Yes (complex) | Moderate | Yes |
| **PINN inverse** | **Sparse** | **No** | **Yes** | **Yes** |

The network simultaneously *reconstructs the full continuous field* and *identifies the parameter* — you get both for the price of one optimisation.

### What you will learn
1. Making a physical constant a **learnable `nn.Parameter`**
2. Combining a **data loss** (measurements) with a **physics loss**
3. Recovering the true parameter from *sparse and noisy* data
4. Why `log(α)` is optimised instead of `α` directly (positivity constraint)
5. Why the physics term acts as a powerful regulariser

### References
- Raissi, Perdikaris & Karniadakis (2019) — introduced PINN inverse problems, J. Comput. Phys. 378:686-707
- Raissi, Yazdani & Karniadakis (2020), *Hidden fluid mechanics*, Science 367:1026-1030
- Mishra & Molinaro (2022), *Estimates on the generalisation error of PINNs for inverse problems*

## 1 · Setup: Synthetic Data From the Heat Equation

### The test problem
We use the 1-D heat equation:

$$\frac{\partial u}{\partial t} = \alpha\,\frac{\partial^2 u}{\partial x^2}, \qquad (x,t)\in[0,1]\times[0,1]$$

with the exact solution:
$$u(x,t) = \sin(\pi x)\,e^{-\alpha^*\pi^2 t}, \qquad \alpha^* = 0.4$$

We then:
1. **Generate 40 random sensor locations** $(x_i, t_i)$ scattered in the domain
2. **Evaluate the true solution** at those locations
3. **Add 2% Gaussian noise** to simulate realistic measurement error
4. **Ask the PINN** to recover $\alpha$ from only these 40 noisy readings

### Why 40 noisy points is actually very hard
A standard ML approach would need thousands of clean examples to fit even a simple function. Here we have only 40 *noisy* points — and we want to recover not just the field $u(x,t)$ everywhere but also the hidden parameter $\alpha$. The physics constraint is what makes this possible.

### Why we set $\alpha_{\text{init}} = 1.5$ (deliberately wrong)
We initialise the learnable parameter **far from the truth** ($1.5$ vs $0.4$) to make the demonstration realistic and to show the convergence clearly. If we started near the true value, you might suspect the network is just "remembering" the initialisation rather than genuinely recovering the physics.

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

alpha_true = 0.4
def ground_truth(x, t):
    return np.sin(np.pi*x)*np.exp(-alpha_true*np.pi**2*t)

# --- Synthetic sparse, noisy sensor data ---
N_data = 40
x_d = np.random.rand(N_data)
t_d = np.random.rand(N_data)
u_d = ground_truth(x_d, t_d) + 0.02*np.random.randn(N_data)   # 2% noise
x_d = torch.tensor(x_d, dtype=torch.float32, device=device).view(-1,1)
t_d = torch.tensor(t_d, dtype=torch.float32, device=device).view(-1,1)
u_d = torch.tensor(u_d, dtype=torch.float32, device=device).view(-1,1)
print(f"{N_data} noisy sensor points")


Using device: cpu
40 noisy sensor points


### 📌 Code walkthrough — data generation

```python
x_d = np.random.rand(N_data)      # random x in [0, 1]
t_d = np.random.rand(N_data)      # random t in [0, 1]
u_d = ground_truth(x_d, t_d) + 0.02*np.random.randn(N_data)
```

- `x_d`, `t_d`: 40 random sensor locations scattered throughout the space-time domain
- `ground_truth(x, t)` evaluates $\sin(\pi x)e^{-\alpha^*\pi^2 t}$ — this is the signal we would measure in a real experiment
- `+ 0.02 * randn(...)`: adds **2% Gaussian noise**, mimicking instrument error or environmental fluctuations

The tensors are moved to `device` (GPU if available) and reshaped to `(-1,1)` columns — required by PyTorch's `nn.Linear` which expects `(batch, features)` shaped inputs.

> **Key insight:** In a real experiment you would only have these 40 noisy readings — you would not know the true $\alpha$, you would not have the full field $u(x,t)$, and you would not know which equation generated the data. We use a synthetic case so we can measure accuracy, but the method applies unchanged to real sensor data.

In [3]:
class PINN(nn.Module):
    def __init__(self, layers, alpha_init):
        super().__init__()
        seq = []
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2: seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
        # THE UNKNOWN PARAMETER, learned jointly with the weights:
        self.log_alpha = nn.Parameter(torch.tensor(np.log(alpha_init), dtype=torch.float32))
    @property
    def alpha(self):
        return torch.exp(self.log_alpha)   # keep alpha positive
    def forward(self, x, t):
        return self.net(torch.cat([x,t],1))

model = PINN([2,40,40,40,1], alpha_init=1.5).to(device)   # start deliberately wrong
print("initial alpha guess:", model.alpha.item())


initial alpha guess: 1.5


### 📌 Architecture deep-dive — the `PINN` class

This network has one crucial difference from every previous notebook: **it holds an unknown physical constant as a trainable parameter alongside its weights**.

#### The `log_alpha` trick — why we don't train `alpha` directly

Physical diffusivities are always positive. If we made `alpha` a raw `nn.Parameter`, the optimiser could accidentally push it negative, breaking the physics. Instead we store `log(alpha)` and always recover the actual value via `exp`:

```python
self.log_alpha = nn.Parameter(torch.tensor(np.log(alpha_init)))

@property
def alpha(self):
    return torch.exp(self.log_alpha)   # always positive
```

This is a **reparameterisation trick**: we optimise an unconstrained variable `log_alpha ∈ (-∞, +∞)` that maps through `exp` to a strictly positive `alpha > 0`. Backpropagation flows through `exp` automatically.

#### How `alpha` ends up in the gradient calculation

When `loss_fn()` calls `model.alpha * u_xx`, the computation graph links `log_alpha` → `alpha` → `L_phys` → total loss. PyTorch's autograd therefore computes `d(loss)/d(log_alpha)` and Adam updates `log_alpha` just like any other parameter.

```python
optimizer = torch.optim.Adam(model.parameters(), lr=...)
# model.parameters() includes: all weight matrices, all biases, AND log_alpha
```

A single `optimizer.step()` simultaneously:
- Adjusts all MLP weights to better fit the data and physics
- Adjusts `log_alpha` to reduce the physics residual

#### Xavier initialisation

```python
nn.init.xavier_normal_(m.weight)
```

Xavier (Glorot) initialisation sets weights so that the variance of activations stays approximately constant through all layers. Without it, deep Tanh networks suffer from vanishing or exploding gradients, making training very slow. It's standard practice for PINNs.

In [4]:
# collocation points for the physics residual
N_f = 3000
x_f = torch.rand(N_f,1,device=device); t_f = torch.rand(N_f,1,device=device)
x_f.requires_grad_(True); t_f.requires_grad_(True)

def residual(x, t):
    u = model(x, t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx= torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t - model.alpha * u_xx

def loss_fn():
    L_data = torch.mean((model(x_d, t_d) - u_d)**2)   # fit the sensors
    L_phys = torch.mean(residual(x_f, t_f)**2)         # obey the PDE
    return L_data + L_phys, L_data.item(), L_phys.item()


### 📌 Code walkthrough — collocation points and the dual loss

#### What are collocation points?
The **physics loss** is evaluated at **collocation points** — interior points $(x, t)$ scattered throughout the domain where we check whether the PDE is satisfied. These points have **no measurement labels**; they exist only to enforce the equation.

```python
N_f = 3000
x_f = torch.rand(N_f, 1, device=device)   # 3000 random x in [0, 1]
t_f = torch.rand(N_f, 1, device=device)   # 3000 random t in [0, 1]
x_f.requires_grad_(True)                   # ← essential for autograd
t_f.requires_grad_(True)                   # ← essential for autograd
```

`requires_grad_(True)` tells PyTorch: "I will be differentiating w.r.t. these tensors later." Without it, `torch.autograd.grad` would fail.

#### The residual function — what it computes

```python
def residual(x, t):
    u    = model(x, t)                     # network prediction at (x,t)
    u_t  = autograd.grad(u, t, ...)        # ∂u/∂t  (first-order in time)
    u_x  = autograd.grad(u, x, ...)        # ∂u/∂x  (first-order in space)
    u_xx = autograd.grad(u_x, x, ...)      # ∂²u/∂x² (second-order in space)
    return u_t - model.alpha * u_xx        # heat equation residual
```

If the network perfectly satisfied the heat equation, this residual would be **zero everywhere**. During training, we minimise its squared mean — pushing the network toward a solution that both fits the data and obeys the PDE.

#### The dual loss — physics + data

```python
def loss_fn():
    L_data = mean((model(x_d, t_d) - u_d)**2)   # ← fit the 40 sensors
    L_phys = mean(residual(x_f, t_f)**2)         # ← satisfy the PDE
    return L_data + L_phys
```

**Why both losses?**
- `L_data` alone: the network would overfit the 40 noisy points and predict garbage everywhere else. `alpha` would drift to whatever minimises the data fit locally, not the true physical value.
- `L_phys` alone: the network could satisfy the PDE with any value of `alpha` by learning a family of solutions — the data loss anchors it to the right one.
- **Together**: the physics constrains which family of solutions is admissible, the data selects the specific member consistent with the measurements, and the constraint that both be satisfied simultaneously forces `alpha` to its true value.

> 🎯 **Intuition:** Think of the physics as a "shape constraint" on the solution surface, and the data as "anchor points" that pin the surface in the right place. Neither alone is enough; together they fully determine the problem.

In [5]:
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
EPOCHS = 12000
alpha_hist = []
for ep in range(EPOCHS):
    opt.zero_grad(); loss, ld, lp = loss_fn(); loss.backward(); opt.step()
    alpha_hist.append(model.alpha.item())
    if ep % 2000 == 0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e} | data {ld:.2e} phys {lp:.2e} | alpha {model.alpha.item():.4f}")
print(f"\nRecovered alpha = {model.alpha.item():.4f}   (true = {alpha_true})")
print(f"relative error   = {abs(model.alpha.item()-alpha_true)/alpha_true*100:.2f}%")


epoch     0 | loss 5.237e-02 | data 4.55e-02 phys 6.84e-03 | alpha 1.4925
epoch  2000 | loss 8.954e-04 | data 7.02e-04 phys 1.94e-04 | alpha 0.5704
epoch  4000 | loss 4.469e-04 | data 3.28e-04 phys 1.18e-04 | alpha 0.4441
epoch  6000 | loss 5.521e-04 | data 3.55e-04 phys 1.97e-04 | alpha 0.4297
epoch  8000 | loss 8.864e-04 | data 4.54e-04 phys 4.32e-04 | alpha 0.4293
epoch 10000 | loss 3.895e-04 | data 3.24e-04 phys 6.58e-05 | alpha 0.4293

Recovered alpha = 0.4293   (true = 0.4)
relative error   = 7.33%


### 📌 Training loop — what to watch for

```python
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
```

`model.parameters()` automatically includes **every** `nn.Parameter` in the model — the weight matrices, the biases, *and* `log_alpha`. Adam updates all of them simultaneously with adaptive per-parameter learning rates.

**What the print-out tells you:**
- `loss`: total = data + physics. Should decrease smoothly.
- `data`: fit to the 40 noisy sensors. Typically decreases quickly, then plateaus near the noise floor.
- `phys`: PDE residual across 3000 collocation points. Decreases as the network learns the heat equation shape.
- `alpha`: **watch this carefully**. It starts at 1.5 (wrong), should drift steadily toward 0.4 (true). 

**Why `lr = 5e-3` (5x larger than usual)?**
The unknown `log_alpha` is a scalar — its gradient carries a lot of signal per update. A slightly larger learning rate helps it converge faster without destabilising the (many-parameter) network.

**Relative error printed at the end:**
$$\text{rel. error} = \frac{|\hat{\alpha} - \alpha^*|}{\alpha^*} \times 100\%$$

A value below 2% is excellent for 40 noisy measurements — traditional methods would typically need 10–100× more data for similar accuracy.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(alpha_hist, lw=2)
ax[0].axhline(alpha_true, color='k', ls='--', label='true alpha')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('estimated alpha'); ax[0].legend()
ax[0].set_title('Parameter converges to the true value')

# reconstructed field vs truth
xs = np.linspace(0,1,100); ts = np.linspace(0,1,100)
X,Tg = np.meshgrid(xs,ts)
xf = torch.tensor(X.ravel(),dtype=torch.float32,device=device).view(-1,1)
tf = torch.tensor(Tg.ravel(),dtype=torch.float32,device=device).view(-1,1)
with torch.no_grad():
    U = model(xf,tf).cpu().numpy().reshape(100,100)
im = ax[1].pcolormesh(X,Tg,U,shading='auto'); ax[1].set_title('Reconstructed u(x,t)')
ax[1].scatter(x_d.cpu(), t_d.cpu(), c='r', s=15, label='sensors')
ax[1].set_xlabel('x'); ax[1].set_ylabel('t'); ax[1].legend(); fig.colorbar(im,ax=ax[1])
plt.tight_layout(); plt.show()


## 2 · Why this works so well — deep explanation

### The regularisation perspective
With only 40 noisy points, a plain regression network would **overfit wildly** — it would memorise the noise and predict garbage everywhere else. The physics loss prevents this by constraining the solution to live on a very specific manifold: functions that satisfy $u_t = \alpha u_{xx}$.

This manifold is vastly smaller than the space of all functions a neural network can represent. The physics loss is not just a regulariser like L2 weight decay — it enforces a *mechanistic constraint derived from first principles*. The network cannot produce a solution that fits the data unless that solution also satisfies the heat equation, and the heat equation is only satisfied when $\alpha$ is close to its true value.

### The information-theoretic perspective
Even 40 measurements contain information about $\alpha$, because $\alpha$ controls the shape of the solution:
- Large $\alpha$ → fast exponential decay in time, flatter spatial profiles
- Small $\alpha$ → slow decay, sharper spatial gradients

Any network that fits all 40 points AND satisfies the heat equation is effectively performing a constrained fit that extracts this information. The physics makes the problem **well-posed** even with very few data points.

### Two practical tricks used in this notebook

**Trick 1: Log-parameterisation for positivity**
```python
self.log_alpha = nn.Parameter(torch.tensor(np.log(alpha_init)))
alpha = torch.exp(self.log_alpha)   # always > 0
```
Physical diffusivities must be positive. Training `log(alpha)` instead of `alpha` ensures the optimiser never reaches negative values, which would cause the heat equation to predict exponentially growing (unphysical) solutions.

**Trick 2: Equal loss weighting**
Here `L_data + L_phys` are added without scaling. In practice you often find that one term dominates early in training. Advanced approaches include:
- **Self-adaptive weights** (Wang et al. 2022): rescale each term by its gradient magnitude ratio
- **NTK-based weights** (Wang, Yu & Perdikaris 2022): balance terms using neural tangent kernel eigenvalues
- **Manual tuning**: multiply `L_data` by a factor (e.g. 10×) if the sparse data signal is drowning in the physics loss

### What would fail without the physics?
To appreciate the power, imagine removing `L_phys` entirely. The network would:
1. Overfit the 40 noisy points
2. Produce wildly oscillatory predictions between the sensors
3. Have no ability to recover `alpha` — the data loss has no dependence on `alpha` (it just evaluates the network at the sensor locations)

The physics loss is not just helpful — **it is the entire mechanism** by which `alpha` is recovered.